<a href="https://colab.research.google.com/github/Timbo3399/INFM_SOTA2/blob/Timo/W1_DataLoader_BPE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Task 1: Data Loader (BPE, Sliding Window, Embeddings)

**Autor:** Timo · **Datum:** 10.10.2026

**Ziel:**
Write a data loader that implements
- Byte Pair Tokenization
- Sliding Window train data creation
- Token embeddings → use PyTorch nn.embedding
- Position Embeddings → use PyTorch nn.embedding
- Test it on the Data Sample...

In [15]:
import os
from importlib.metadata import version

import requests
import tiktoken
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

print("torch", version("torch"), "| tiktoken", version("tiktoken"))  # Versionen für Reproduzierbarkeit

tokenizer = tiktoken.get_encoding("gpt2")

torch 2.11.0+cpu | tiktoken 0.14.0


In [16]:
# Alle Parameter an einer Stelle – keine versteckten Defaults in Funktionen
CONFIG = {
    "data_path": "data/The-lord-of-the-rings.txt",
    "seed": 123,
    "max_length": 4,     # Fenstergröße = Kontextlänge in Tokens
    "stride": 4,         # Schrittweite des Fensters (= max_length → keine Überlappung)
    "batch_size": 8,
    "output_dim": 256,   # Länge eines Embedding-Vektors
}

In [17]:
file_path = CONFIG["data_path"]
os.makedirs(os.path.dirname(file_path), exist_ok=True)

if not os.path.exists(file_path):
    url = (
        "https://dn721603.ca.archive.org/0/items/"
        "tolkien-j.-the-lord-of-the-rings-harper-collins-ebooks-2010/"
        "Tolkien-J.-The-lord-of-the-rings-HarperCollins-ebooks-2010_djvu.txt"
    )
    response = requests.get(url, timeout=30)
    response.raise_for_status()
    with open(file_path, "wb") as f:
        f.write(response.content)

In [18]:
enc_text = tokenizer.encode(raw_text, allowed_special={"<|endoftext|>"})
print("Anzahl Tokens:", len(enc_text))
assert tokenizer.decode(enc_text) == raw_text   # BPE ist verlustfrei: Text → IDs → Text

Anzahl Tokens: 928088


In [19]:
print("Vokabulargröße:", tokenizer.n_vocab)
ids = tokenizer.encode("Hobbitses sneaked into Mordor")
print(ids)
print([tokenizer.decode([i]) for i in ids])   # unbekannte Wörter → bekannte Teilstücke

Vokabulargröße: 50257
[39, 672, 9895, 274, 10505, 4335, 656, 29548, 273]
['H', 'ob', 'bits', 'es', ' sne', 'aked', ' into', ' Mord', 'or']


In [20]:
context_size = 4

x = enc_sample[:context_size]
y = enc_sample[1:context_size+1]

for i in range(1, context_size + 1):
    print(tokenizer.decode(enc_sample[:i]), "--->", tokenizer.decode([enc_sample[i]]))
assert x[1:] == y[:-1]   # y = x um 1 Token verschoben


 ---> Three

Three --->  Rings

Three Rings --->  for

Three Rings for --->  the
x: [198, 12510, 26028, 329]
y:      [12510, 26028, 329, 262]


In [21]:
class SlidingWindowDataset(Dataset):
    """Zerlegt Text in (input, target)-Fenster; target = input um 1 Token verschoben."""

    def __init__(self, txt, tokenizer, max_length, stride):
        token_ids = tokenizer.encode(txt)
        assert len(token_ids) > max_length, (
            f"Text hat nur {len(token_ids)} Tokens, braucht > max_length={max_length}")

        self.input_ids, self.target_ids = [], []
        for i in range(0, len(token_ids) - max_length, stride):
            self.input_ids.append(torch.tensor(token_ids[i:i + max_length]))
            self.target_ids.append(torch.tensor(token_ids[i + 1:i + max_length + 1]))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]

In [22]:
def create_dataloader(txt, tokenizer, batch_size, max_length, stride, shuffle, seed):
    dataset = SlidingWindowDataset(txt, tokenizer, max_length, stride)
    assert len(dataset) >= batch_size, f"Nur {len(dataset)} Fenster für batch_size={batch_size}"
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=True,                                   # alle Batches gleich groß
        generator=torch.Generator().manual_seed(seed),    # reproduzierbares Shuffling
    )

In [23]:
demo_loader = create_dataloader(raw_text[:5_000], tokenizer, batch_size=1,
                                max_length=4, stride=1, shuffle=False, seed=CONFIG["seed"])
it = iter(demo_loader)
(x1, y1), (x2, y2) = next(it), next(it)
print(x1, y1)
print(x2, y2)
assert torch.equal(x1[0, 1:], x2[0, :-1])   # stride=1 → Fenster rückt genau 1 Token weiter

tensor([[48294,  7054,  1268,   327]]) tensor([[7054, 1268,  327, 5868]])
tensor([[7054, 1268,  327, 5868]]) tensor([[ 1268,   327,  5868, 12964]])


In [24]:
dataloader = create_dataloader(raw_text, tokenizer, CONFIG["batch_size"], CONFIG["max_length"],
                               CONFIG["stride"], shuffle=False, seed=CONFIG["seed"])
inputs, targets = next(iter(dataloader))
print("Anzahl Batches:", len(dataloader))
print(inputs)
print(targets)
assert inputs.shape == (CONFIG["batch_size"], CONFIG["max_length"])
assert torch.equal(inputs[:, 1:], targets[:, :-1])

Anzahl Batches: 29002
tensor([[48294,  7054,  1268,   327],
        [ 5868, 12964,  4805,  1268],
        [ 6242,  2390,    33,  3861],
        [   36,   220,   628,   198],
        [   50,   220,   628,   198],
        [  464,  4453,   286,   262],
        [26028,   220,   628,   198],
        [17513,   220,   628,   198]])
tensor([[ 7054,  1268,   327,  5868],
        [12964,  4805,  1268,  6242],
        [ 2390,    33,  3861,    36],
        [  220,   628,   198,    50],
        [  220,   628,   198,   464],
        [ 4453,   286,   262, 26028],
        [  220,   628,   198, 17513],
        [  220,   628,   198,    41]])


In [25]:
torch.manual_seed(CONFIG["seed"])
mini = nn.Embedding(6, 3)                       # 6 Token-IDs, je ein 3-dim. Vektor
idx = torch.tensor([3])
print(mini.weight)
print(mini(idx))
assert torch.equal(mini(idx)[0], mini.weight[3])                      # Lookup = Zeile 3
one_hot = nn.functional.one_hot(idx, num_classes=6).float()
assert torch.allclose(one_hot @ mini.weight, mini(idx))               # = One-Hot × Gewichtsmatrix

Parameter containing:
tensor([[ 0.3374, -0.1778, -0.1690],
        [ 0.9178,  1.5810,  1.3010],
        [ 1.2753, -0.2010, -0.1606],
        [-0.4015,  0.9666, -1.1481],
        [-1.1589,  0.3255, -0.6315],
        [-2.8400, -0.7849, -1.4096]], requires_grad=True)
tensor([[-0.4015,  0.9666, -1.1481]], grad_fn=<EmbeddingBackward0>)


In [26]:
vocab_size = tokenizer.n_vocab   # aus dem Tokenizer, nicht hart codiert
torch.manual_seed(CONFIG["seed"])
token_embedding_layer = nn.Embedding(vocab_size, CONFIG["output_dim"])
token_embeddings = token_embedding_layer(inputs)
assert token_embeddings.shape == (CONFIG["batch_size"], CONFIG["max_length"], CONFIG["output_dim"])

In [27]:
context_length = CONFIG["max_length"]   # eine Zeile pro Position im Fenster
pos_embedding_layer = nn.Embedding(context_length, CONFIG["output_dim"])
pos_embeddings = pos_embedding_layer(torch.arange(context_length))
assert pos_embeddings.shape == (context_length, CONFIG["output_dim"])

In [28]:
input_embeddings = token_embeddings + pos_embeddings   # [8,4,256] + [4,256] → Broadcasting über Batch
assert input_embeddings.shape == token_embeddings.shape
print(input_embeddings.shape)

torch.Size([8, 4, 256])
